In [17]:
import sys
import warnings
from typing import cast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from sklearn.model_selection import train_test_split

sys.path.append(r"../baseline_v2")
from add_features import add_modified_features
from load_data import load_data
from preprocess import build_preprocessor, build_target_transformer

%matplotlib inline

warnings.filterwarnings('ignore')

## Data Preprocessing

### load

In [18]:
train, test = load_data()
train.shape, test.shape

((1458, 80), (1459, 79))

### prepare

In [19]:
X = train.copy()
y = X.pop('SalePrice')
X_test = test.copy()

### build pipeline

In [20]:
log_standardize_y = build_target_transformer()
tree_preprocessor, reg_preprocessor = build_preprocessor()

### add modified -> drop useless -> filter outlier ->train test split

In [21]:
# add modified
X = add_modified_features(pl.DataFrame(X))
X_test = add_modified_features(pl.DataFrame(X_test))
X.index = y.index

# split
X_train, X_va, y_train, y_va = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape, X_va.shape, y_train.shape, y_va.shape

((1166, 109), (292, 109), (1166,), (292,))

In [22]:
s = X_train.isnull().sum()
s[s > 0]

Series([], dtype: int64)

In [23]:
transformed_X_train = reg_preprocessor.fit_transform(X_train,y_train)
transformed_X_train = cast(np.ndarray,transformed_X_train)
transformed_X_train_df = pd.DataFrame(data=transformed_X_train,columns=reg_preprocessor.get_feature_names_out())
transformed_X_train_df.shape

(1166, 73)

In [ ]:

transformed_X_train_df.hist(figsize=(12,90), bins=50,layout=(30,3))
plt.tight_layout()

In [ ]:
threshold = 0.7

corr = transformed_X_train_df.corr()

# 対角成分と重複する組み合わせを除外
upper_mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)

high_corr = (
    corr.where(upper_mask)
    .stack()
    .rename("correlation")
    .reset_index()
    .rename(columns={
        "level_0": "feature_1",
        "level_1": "feature_2",
    })
)

# 相関係数の絶対値でフィルタ
high_corr = (
    high_corr.loc[high_corr["correlation"].abs() >= threshold]
    .assign(abs_correlation=lambda df: df["correlation"].abs())
    .sort_values("abs_correlation", ascending=False)
    .reset_index(drop=True)
)

high_corr

,feature_1,feature_2,correlation,abs_correlation
0,one_hot_encode__LotConfig_CulDSac,remainder__IsCuldsac,1.000000,1.000000
1,ordinal_encode__FireplaceQu,scale__FireplaceScore,0.976581,0.976581
2,ordinal_encode__GarageQual,ordinal_encode__GarageCond,0.959495,0.959495
3,ordinal_encode__GarageCond,remainder__NoGarage,-0.953733,0.953733
4,ordinal_encode__GarageQual,remainder__NoGarage,-0.945421,0.945421
5,scale__GarageCars,scale__GarageArea,0.889450,0.889450
6,log__LivArea_x_Qual,scale__TotalFlrSF,0.855825,0.855825
7,scale__GarageYrBlt,scale__BuildingAge,-0.852626,0.852626
8,target_encode__MasVnrType,log__MasVnrArea,0.848796,0.848796
9,scale__KitchenAbvGr,remainder__HasTwoFamilies,0.826551,0.826551


In [ ]:
high_corr.iloc[11:16,:-1]

,feature_1,feature_2,correlation
11,eq_val_label__PavedDrive,scale__MissingNormalyUtilsCount,-0.803600
12,target_encode__Neighborhood,remainder__ExNeighborhoods,0.798013
13,scale__TotalFlrSF,scale__BsmtScore,0.792869
14,log__LotArea,log__LivLotRatio,-0.758796
15,one_hot_encode__LotConfig_Corner,one_hot_encode__LotConfig_Inside,-0.738174


In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

threshold = 10  # VIFの閾値(5または10が一般的)

# VIF計算用に定数項を追加(切片ぶんを除いてVIFを正しく算出するため)
X_vif = transformed_X_train_df.assign(const=1)

vif = pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ],
})

# 定数項の行を除外
vif = vif.loc[vif["feature"] != "const"].reset_index(drop=True)

# VIFの高い順にソートし、閾値でフィルタ
high_vif = (
    vif.loc[vif["VIF"] >= threshold]
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

high_vif

,feature,VIF
0,eq_val_label__PavedDrive,inf
1,one_hot_encode__LotConfig_Corner,inf
2,one_hot_encode__LotConfig_CulDSac,inf
3,one_hot_encode__LotConfig_Inside,inf
4,one_hot_encode__LotConfig_infrequent_sklearn,inf
5,one_hot_encode__LotConfig_FR2,inf
6,remainder__NoGarage,inf
7,scale__MissingNormalyUtilsCount,inf
8,remainder__IsCuldsac,inf
9,remainder__NoBsmt,inf
